# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShivanshRastogi315/FlyRank_Internship-repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method Choice: Random Forest Classifier (Supervised)**
Initially, I hypothesized that AI-traffic pages were mathematical anomalies and tested an unsupervised Isolation Forest. It failed (Precision@20 of 0%), proving that AI traffic is not merely an outlier—it is a specific, learnable pattern. I pivoted to a supervised **Random Forest Classifier**. Because our dataset is highly imbalanced (~6.5% positive rate), I applied `class_weight='balanced'` to heavily penalize the model when it misses the rare AI traffic events.

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# Load data and define target
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_ai_traffic'] = (df['ai_sessions_90d'] > 0).astype(int)

# Fill nulls
df['search_volume'] = df['search_volume'].fillna(0)
df['avg_position'] = df['avg_position'].fillna(100)
df['word_count'] = df['word_count'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)
df['days_since_last_update'] = df['days_since_last_update'].fillna(0)
df['engagement_rate'] = df['engagement_rate'].fillna(0)

# Define Features
features = ['search_volume', 'avg_position', 'word_count', 
            'content_age_days', 'days_since_last_update', 'engagement_rate']

print("Supervised Method initialized and features staged.")

Supervised Method initialized and features staged.


## 2. Split design

**Split Design: GroupShuffleSplit (Client-Aware Holdout)**
A random train/test split would be dishonest because pages from the same client share domain authority and architecture. If we put Client A's pages in both train and test, the model will just memorize Client A's domain authority (data leakage). Instead, we use `GroupShuffleSplit` on `client_id` to ensure the model is tested on entirely unseen clients, proving its ability to generalize to new business properties.

In [11]:
# GroupShuffleSplit based on client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Generate indices for train and test
for train_idx, test_idx in gss.split(df, groups=df['client_id']):
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

# Scale numerical features for the Isolation Forest
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])

y_test = test_df['is_ai_traffic']

print(f"Training Set: {len(train_df)} rows")
print(f"Test Set (Unseen Clients): {len(test_df)} rows")

Training Set: 23837 rows
Test Set (Unseen Clients): 6163 rows


## 3. Train + compare vs my baseline

**Training & Baseline Comparison:**
The Isolation Forest is trained exclusively on the feature matrix. Once trained, we generate anomaly scores for the test set, rank them descending (highest anomaly score first), and calculate the Precision@20 to compare directly against the 0.00% achieved by our ML-07 heuristic.

In [12]:
# 1. Train the Supervised Random Forest
# class_weight='balanced' is the secret weapon for imbalanced datasets
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, max_depth=10)
rf_model.fit(X_train, train_df['is_ai_traffic']) # We are now feeding it the answers!

# 2. Score the Test Set (We want the probability of it being AI traffic)
# predict_proba returns [prob_class_0, prob_class_1]. We want index 1.
test_df['ml_confidence_score'] = rf_model.predict_proba(X_test)[:, 1]

# 3. Rank and pull Top 20
ranked_test = test_df.sort_values(by='ml_confidence_score', ascending=False)
top_20_ml = ranked_test.head(20)

# 4. Calculate Precision@20
ml_true_positives = (top_20_ml['ai_sessions_90d'] > 0).sum()
ml_precision_at_20 = (ml_true_positives / 20) * 100

print("--- PERFORMANCE COMPARISON ---")
print(f"ML-07 Baseline Precision@20:   0.00%")
print(f"ML-08 Random Forest Precision@20: {ml_precision_at_20:.2f}%")

if ml_precision_at_20 > 0:
    print(f"\n✅ SUCCESS: The Supervised ML model defeated the baseline, finding {ml_true_positives} true winners!")

--- PERFORMANCE COMPARISON ---
ML-07 Baseline Precision@20:   0.00%
ML-08 Random Forest Precision@20: 15.00%

✅ SUCCESS: The Supervised ML model defeated the baseline, finding 3 true winners!


## 4. Errors and interpretation

**Error Interpretation:**
The Supervised Random Forest successfully defeated the baseline heuristic, achieving a 15% Precision@20 (compared to the baseline's 0%). By applying balanced class weights, the model learned to look beyond raw search volume and identify the deeper structural patterns of AI traffic. However, it still produced 17 False Positives in the top 20. These errors occur because the model relies heavily on proxy metrics (like word count and freshness); the false positives perfectly mimic the "winning" structural pattern, but likely lack unmeasured latent variables—such as specific topical intent or off-page domain authority—that actually trigger the AI Overviews in the real world.

In [13]:
# Identify False Positives in the Top 20 (High ML Score, but ZERO AI Traffic)
false_positives = top_20_ml[top_20_ml['is_ai_traffic'] == 0]

print("--- ERROR ANALYSIS (False Positives in Top 20) ---")
print(f"Total Errors in Top 20: {len(false_positives)}")

if not false_positives.empty:
    # Notice we swapped 'anomaly_score' for 'ml_confidence_score' here
    display(false_positives[['content_id', 'ml_confidence_score', 'search_volume', 'word_count', 'engagement_rate', 'ai_sessions_90d']].head())

--- ERROR ANALYSIS (False Positives in Top 20) ---
Total Errors in Top 20: 17


,content_id,ml_confidence_score,search_volume,word_count,engagement_rate,ai_sessions_90d
2332,content_2841dad7e57a,0.775133,0.0,2668.0,14.29,0
27648,content_355c44a1ed2c,0.729109,0.0,2754.0,6.67,0
961,content_05688a1937fe,0.726874,10.0,3008.0,2.17,0
22468,content_d263c7f4f8c0,0.715548,10.0,2984.0,5.88,0
6740,content_3b36899453fe,0.699481,0.0,2172.0,4.76,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.